In [1]:
%load_ext autoreload
%autoreload 2

from tasks.diffusion import GaussianDiffusionTask
from tasks.autoencoder import AETask
import os
import torch
from tqdm import tqdm
from model.gaussian_diffusion import *
from evaluate import load
import einx

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["LATENT_CONTROL_CKPT_DIR"] = (
    "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints"
)

In [3]:
ae_task = AETask.load_from_checkpoint(
            os.path.join(
                "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints/cm9ujm08/"
                "last.ckpt",
            ),
            strict=False
        )

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.


In [4]:
ae_task = ae_task.cuda()

In [5]:
ae_task.setup()

In [6]:
batch = next(iter(ae_task.val_dataloader()))

In [7]:
with torch.no_grad():
    z = ae_task.encoder(
        input_ids=batch["input_ids_enc"].cuda(), 
        attention_mask=batch["attention_mask_enc"].cuda()
    )

In [8]:
z = z[:10]

In [21]:
position_ids = torch.zeros(
    z.shape[0], z.shape[1] + 1, device=z.device, dtype=torch.long
)
bos_emb = ae_task.decoder.backbone.get_input_embeddings()(
    torch.tensor(ae_task.decoder.tokenizer.bos_token_id).cuda()
)
bos_emb = einx.rearrange("d -> b 1 d", bos_emb, b=z.shape[0])
z_with_bos = torch.cat([z, bos_emb], dim=1).half()

In [25]:
with torch.no_grad():
    output = ae_task.decoder.backbone.generate(
        inputs_embeds=z_with_bos,
        #position_ids=position_ids,
        max_length=200,
        do_sample=True,
        top_p=0.92,
        top_k=50,
        num_beams=1,
        temperature=0.9,
        return_dict_in_generate=True,
        use_cache=True,
        pad_token_id=ae_task.decoder.tokenizer.pad_token_id,
        eos_token_id=ae_task.decoder.tokenizer.eos_token_id,
    )

In [26]:
seqs = ae_task.decoder.tokenizer.batch_decode(output.sequences, skip_special_tokens=True)

In [27]:
seqs

[' They They had to to to to to to to. to. When had called me and she called her. She said thank you. She said goodbye in housing.',
 ' my my taught me when when. I has. My teacher began to recently. This continues to. This is.',
 ' Tom took a a a a         ',
 ' Martin heard heard a he he he he he he he he he he he he he he had seen a man staring at mask at clown..',
 ' he he he he t he . . . t . t . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . a . . a . . . a . a . a . a . a 1 . a . a C TOTTERLOTTERLOTTERLOTTERLOTTERLOTTERLOTTERLTERLTERLTERLTERLTERLTERLTERLTERLTERLTERLTERLTATERL TOM L T L TAKE TOM L T TATERL T TA MAN TTA TE TOM TOM TOM TOM L T T TTA',
 "Thethe'whenthe the when the to when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when when whe

In [50]:
prefill = ae_task.decoder.backbone(
    inputs_embeds=z.half(),
    use_cache=True,
)
cache = prefill.past_key_values
cache_position = torch.tensor([0])
bos = torch.full(
    (z.shape[0], 1),
    ae_task.decoder.tokenizer.bos_token_id,
    device=z.device,
    dtype=torch.long,
)

In [51]:
with torch.no_grad():
    output = ae_task.decoder.backbone.generate(
        input_ids=bos,
        past_key_values=cache,
        cache_position=cache_position,
        max_length=200,
        do_sample=True,
        top_p=0.92,
        top_k=50,
        num_beams=1,
        temperature=0.9,
        return_dict_in_generate=True,
        use_cache=True,
        pad_token_id=ae_task.decoder.tokenizer.pad_token_id,
        eos_token_id=ae_task.decoder.tokenizer.eos_token_id,
        )

In [49]:
seqs = ae_task.decoder.tokenizer.batch_decode(output.sequences, skip_special_tokens=True)
seqs

[' they they had to to to to to to to. They And When He There She You She I I I C My advice said said housing was was was. This was was was.',
 ' My grandmother taught me when. I my had. My grandmother had done this projects. I continue to projects. This created.',
 ' Tom took a a a          ',
 ' Martin heard heard a he he he he he he he he he he he he he had a look out of clown face in clown face..',
 ' he he he he          in K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K K',
 'TheWhen the the the when the to the the to to to to to.. to. to. to. to, they with ( 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 48 

In [9]:
diffusion_task = GaussianDiffusionTask.load_from_checkpoint(
            os.path.join(
                "/network/scratch/l/leo.gagnon/sentence_diffusion/logs/checkpoints/a1pq0e97/"
                "last.ckpt",
            ),
            strict=False,
        )

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/peft/tuners/lora/layer.py:2156: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [10]:
diffusion_task = diffusion_task.cuda()

In [11]:
diffusion_task.setup()

In [12]:
mauve = diffusion_task.get_mauve_score()

Featurizing q:  88%|████████▊ | 7/8 [00:04<00:00,  1.67it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 214.00 MiB. GPU 0 has a total capacity of 44.64 GiB of which 118.44 MiB is free. Including non-PyTorch memory, this process has 44.52 GiB memory in use. Of the allocated memory 39.63 GiB is allocated by PyTorch, and 4.36 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [7]:
mauve

0.11225014485875065

In [1]:
from transformers.models.auto.modeling_auto import AutoModelForCausalLM
from transformers.models.auto.tokenization_auto import AutoTokenizer


tok = AutoTokenizer.from_pretrained('thesephist/contra-bottleneck-t5-large-wikipedia', model_max_length=512)
model = AutoModelForCausalLM.from_pretrained('thesephist/contra-bottleneck-t5-large-wikipedia', trust_remote_code=True)

/home/mila/l/leo.gagnon/sentence_diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
A new version of the following files was downloaded from https://huggingface.co/thesephist/contra-bottleneck-t5-large-wikipedia:
- bottleneck_t5.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Instantiating a decoder T5Attention without passing `layer_idx` is not recommended and will to errors during the forward call, if caching is used. Please make sure to provide a `layer_idx` when creating this class.


In [8]:
list(model.modules())[3]

ModuleList(
  (0): T5Block(
    (layer): ModuleList(
      (0): T5LayerSelfAttention(
        (SelfAttention): T5Attention(
          (q): Linear(in_features=1024, out_features=1024, bias=False)
          (k): Linear(in_features=1024, out_features=1024, bias=False)
          (v): Linear(in_features=1024, out_features=1024, bias=False)
          (o): Linear(in_features=1024, out_features=1024, bias=False)
          (relative_attention_bias): Embedding(32, 16)
        )
        (layer_norm): T5LayerNorm()
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (1): T5LayerFF(
        (DenseReluDense): T5DenseGatedActDense(
          (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
          (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
          (wo): Linear(in_features=2816, out_features=1024, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
          (act): NewGELUActivation()
        )
        (layer_norm): T5LayerNorm()
        (d

In [79]:
import wandb
from typing import Any, List, Optional, Dict

def get_runs_by_config(
    entity: str,
    project: str,
    config_filters: Dict[str, Any],
    state: Optional[str] = None
) -> List[str]:
    """
    Retrieve run IDs from a W&B project that match specific config values.
    
    Args:
        entity: W&B entity name
        project: W&B project name
        config_filters: Dictionary of config key-value pairs to filter by
                       e.g., {"sweep_id": "dae_sweep", "task.diffusion.model_type": "transformer"}
        state: Optional run state filter ("finished", "running", "crashed", etc.)
    
    Returns:
        List of run IDs that match the criteria
    """
    api = wandb.Api()
    
    # Build the filter dictionary for the API
    filters = {}
    if state:
        filters["state"] = state
    
    # Get all runs first, then filter manually since W&B's config filtering can be unreliable
    runs = api.runs(f"{entity}/{project}", filters=filters)
    
    matching_run_ids = []
    
    for run in runs:
        # Check if all config filters match
        matches_all = True
        for config_key, expected_value in config_filters.items():
            # Navigate nested config using dot notation
            config_value = run.config
            for key_part in config_key.split('.'):
                if isinstance(config_value, dict) and key_part in config_value:
                    config_value = config_value[key_part]
                else:
                    config_value = None
                    break
            
            if config_value != expected_value:
                matches_all = False
                break
        
        if matches_all:
            matching_run_ids.append(run.id)
    
    return matching_run_ids

In [82]:
def list_run_ids(
    entity: str,
    project: str,
    where: Optional[Dict[str, Any]] = None,
    state: Optional[str] = None,
) -> List[str]:
    api = wandb.Api()
    filters: Dict[str, Any] = {}

    # Convert dot-paths to "config.<dotpath>" for W&B filters
    if where:
        for k, v in where.items():
            filters[f"config.{k}"] = v

    if state:
        filters["state"] = state  # e.g., "finished", "failed", "running", etc.

    runs = api.runs(f"{entity}/{project}", filters=filters)
    return [r.id for r in runs]

In [91]:
api = wandb.Api()

In [ ]:
filters = {
    "config.sweep_id": "dae_sweep",
}
runs = api.runs(f"guillaume-lajoie/sentence_diffusion", filters=filters)
ids = [run.id for run in runs]

In [98]:
ae_task.decoder.tokenizer.bos_token_id, ae_task.decoder.tokenizer.eos_token_id, ae_task.decoder.tokenizer.pad_token_id

(50256, 50256, 50257)

In [ ]:
list_run_ids(
    entity="guillaume-lajoie",
    project="sentence_diffusion",
    filters={
        "config.sweep_id": "dae_sweep",
    },
    state="finished"
)

[]

In [87]:
run_ids = get_runs_by_config(
    entity="guillaume-lajoie",
    project="sentence_diffusion",
    config_filters={
        "sweep_id": {"$in": ["dae_sweep"]},  # Has diffusion task
    },
    state="finished"
)

In [88]:
run_ids

[]